In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into memory (variables kept for follow-up)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Helper to compute overview: row_count, min_timestamp, max_timestamp (as tz-aware ISO strings)
def overview(df, ts_col="timestamp"):
    row_count = df.shape[0]
    if row_count == 0:
        min_ts = None
        max_ts = None
    else:
        ts_series = pd.to_datetime(df[ts_col], unit='s', utc=True)
        ts_series = ts_series.dt.tz_convert(tz)
        min_ts = ts_series.min().isoformat()
        max_ts = ts_series.max().isoformat()
    return pd.DataFrame([{"row_count": row_count, "min_timestamp": min_ts, "max_timestamp": max_ts}])

# Helper to get top cmdb_id counts (top N)
def top_cmdb_counts(df, top_n=20):
    if "cmdb_id" not in df.columns or df.shape[0] == 0:
        return pd.DataFrame(columns=["cmdb_id", "count"])
    s = df["cmdb_id"].value_counts().head(top_n).reset_index()
    s.columns = ["cmdb_id", "count"]
    return s

# Helper to get top name counts for a given column
def top_name_counts(df, col_name, top_n=20):
    if col_name not in df.columns or df.shape[0] == 0:
        return pd.DataFrame(columns=[col_name, "count"])
    s = df[col_name].value_counts().head(top_n).reset_index()
    s.columns = [col_name, "count"]
    return s

# Prepare summaries for each file
metric_overview = overview(metric_df, "timestamp")
metric_cmdb_top = top_cmdb_counts(metric_df, top_n=20)
metric_kpi_top = top_name_counts(metric_df, "kpi_name", top_n=20)

trace_overview = overview(trace_df, "timestamp")
trace_cmdb_top = top_cmdb_counts(trace_df, top_n=20)
trace_name_top = top_name_counts(trace_df, "trace_name", top_n=20)

log_overview = overview(log_df, "timestamp")
log_cmdb_top = top_cmdb_counts(log_df, top_n=20)
log_name_top = top_name_counts(log_df, "log_name", top_n=20)

error_overview = overview(error_logs_df, "timestamp")
error_cmdb_top = top_cmdb_counts(error_logs_df, top_n=20)
# examples: top 10 rows with timestamp converted to tz-aware ISO, plus cmdb_id and message
if error_logs_df.shape[0] > 0:
    err_examples = error_logs_df.copy()
    err_examples["timestamp"] = pd.to_datetime(err_examples["timestamp"], unit='s', utc=True).dt.tz_convert(tz).astype(str)
    error_examples = err_examples[["timestamp", "cmdb_id", "message"]].head(10)
else:
    error_examples = pd.DataFrame(columns=["timestamp", "cmdb_id", "message"])

# Final compact outputs (small summaries / tops). These variables are the returned objects.
metric_overview, metric_cmdb_top, metric_kpi_top, trace_overview, trace_cmdb_top, trace_name_top, log_overview, log_cmdb_top, log_name_top, error_overview, error_cmdb_top, error_examples

```
Out[1]:
```
summary = (
    "Summary of telemetry inventories (compact):\n\n"
    "metric.csv:\n"
    "- rows: 319,703\n"
    "- time range (UTC+8): 2022-03-20T10:00:00 → 2022-03-20T11:00:00\n"
    "- top cmdb_id (top 20 shown): adservice (20,801), adservice2 (20,313), frontend2-0 (11,956), "
    "frontend-2 (11,468), frontend-1 (11,468), frontend-0 (11,468), checkoutservice-1 (8,540), "
    "checkoutservice2-0 (8,540), checkoutservice-2 (8,296), checkoutservice-0 (8,296), "
    "productcatalogservice-1 (7,320), productcatalogservice-0 (7,320), productcatalogservice-2 (7,076), "
    "productcatalogservice2-0 (6,344), recommendationservice2-0 (6,222), cartservice-2 (6,100), cartservice-0 (6,100), "
    "cartservice2-0 (6,100), recommendationservice-1 (5,978), recommendationservice-2 (5,978).\n"
    "- top KPI names (top 20): primarily container.node-6.* metrics (network receive/transmit packets/MB, errors), "
    "plus container.node-6 memory, fs, and task metrics (many entries at 2,196 and 2,074 counts).\n\n"
    "trace.csv:\n"
    "- rows: 36,484\n"
    "- time range (UTC+8): 2022-03-20T10:30:00 → 2022-03-20T11:00:00\n"
    "- top cmdb_id (top 20 shown): frontend-2 (2,748), frontend-1 (2,744), frontend-0 (2,716), checkoutservice-1 (2,712), "
    "checkoutservice-2 (2,676), checkoutservice-0 (2,652), productcatalogservice-2 (1,200), productcatalogservice-1 (1,200), "
    "productcatalogservice-0 (1,196), frontend2-0 (1,084), checkoutservice2-0 (1,080), recommendationservice-0 (840), "
    "recommendationservice-2 (840), cartservice-0 (840), cartservice-2 (840), cartservice-1 (840), recommendationservice-1 (840), "
    "shippingservice-1 (720), shippingservice-0 (720), shippingservice-2 (720).\n"
    "- top trace names (top 20): dominated by traces from frontends and checkoutservice, e.g. "
    "trace.from_frontend-2.row_count, .error_rate, .duration_p95, .duration_mean (each 627), and similar per-frontend/checkouts (600+ counts).\n\n"
    "log.csv:\n"
    "- rows: 1,852\n"
    "- time range (UTC+8): 2022-03-20T10:30:00 → 2022-03-20T11:00:00\n"
    "- top cmdb_id (top 20 shown): many services each with ~60–62 rows (e.g., adservice-2, cartservice-1, cartservice-0, cartservice-2, currencyservice-1, ...).\n"
    "- top log names: log.error_count (926), log.row_count (926).\n\n"
    "error_logs.csv:\n"
    "- rows: 0 (no error log rows available in this file for the time window)\n\n"
    "Notes:\n"
    "- All times are reported in UTC+8. The metric dataset is the largest and contains many node-6 container metrics (network/memory/fs). "
    "- Trace data is centered on frontend and checkoutservice traces. Log data contains aggregated error_count/row_count metrics. "
    "- No raw time series are shown here; full tables are retained in kernel variables for follow-up analysis."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   row_count              min_timestamp              max_timestamp
0     319703  2022-03-20T10:00:00+08:00  2022-03-20T11:00:00+08:00,                      cmdb_id  count
0                  adservice  20801
1                 adservice2  20313
2                frontend2-0  11956
3                 frontend-2  11468
4                 frontend-1  11468
5                 frontend-0  11468
6          checkoutservice-1   8540
7         checkoutservice2-0   8540
8          checkoutservice-2   8296
9          checkoutservice-0   8296
10   productcatalogservice-1   7320
11   productcatalogservice-0   7320
12   productcatalogservice-2   7076
13  productcatalogservice2-0   6344
14  recommendationservice2-0   6222
15             cartservice-2   6100
16             cartservice-0   6100
17            cartservice2-0   6100
18   recommendationservice-1   5978
19   recommendationservice-2   5978,                                              kpi_name  count
0   container.node-6.container_network_receive_pac...   2196
1   container.node-6.container_network_transmit_pa...   2196
2   container.node-6.container_network_receive_MB....   2196
3   container.node-6.container_network_receive_pac...   2196
4   container.node-6.container_network_receive_err...   2196
5   container.node-6.container_network_transmit_er...   2196
6   container.node-6.container_network_transmit_pa...   2196
7   container.node-6.container_network_transmit_MB...   2196
8              container.node-6.container_memory_swap   2074
9       container.node-6.container_fs_reads./dev/vda1   2074
10   container.node-6.container_memory_working_set_MB   2074
11     container.node-6.container_fs_inodes./dev/vda1   2074
12        container.node-6.container_file_descriptors   2074
13     container.node-6.container_memory_max_usage_MB   2074
14              container.node-6.container_memory_rss   2074
15         container.node-6.container_memory_usage_MB   2074
16    container.node-6.container_tasks_state.sleeping   2074
17     container.node-6.container_tasks_state.running   2074
18   container.node-6.container_tasks_state.iowaiting   2074
19  container.node-6.container_fs_sector_reads./de...   2074,    row_count              min_timestamp              max_timestamp
0      36484  2022-03-20T10:30:00+08:00  2022-03-20T11:00:00+08:00,                     cmdb_id  count
0                frontend-2   2748
1                frontend-1   2744
2                frontend-0   2716
3         checkoutservice-1   2712
4         checkoutservice-2   2676
5         checkoutservice-0   2652
6   productcatalogservice-2   1200
7   productcatalogservice-1   1200
8   productcatalogservice-0   1196
9               frontend2-0   1084
10       checkoutservice2-0   1080
11  recommendationservice-0    840
12  recommendationservice-2    840
13            cartservice-0    840
14            cartservice-2    840
15            cartservice-1    840
16  recommendationservice-1    840
17        shippingservice-1    720
18        shippingservice-0    720
19        shippingservice-2    720,                                     trace_name  count
0              trace.from_frontend-2.row_count    627
1             trace.from_frontend-2.error_rate    627
2           trace.from_frontend-2.duration_p95    627
3          trace.from_frontend-2.duration_mean    627
4              trace.from_frontend-1.row_count    626
5             trace.from_frontend-1.error_rate    626
6           trace.from_frontend-1.duration_p95    626
7          trace.from_frontend-1.duration_mean    626
8          trace.from_frontend-0.duration_mean    619
9           trace.from_frontend-0.duration_p95    619
10            trace.from_frontend-0.error_rate    619
11             trace.from_frontend-0.row_count    619
12  trace.from_checkoutservice-1.duration_mean    529
13   trace.from_checkoutservice-1.duration_p95    529
14     trace.from_checkoutservice-1.error_rate    529
15      trace.from_checkoutservice-1.row_count    529
16      trace.from_checkoutservice-2.row_count    526
17     trace.from_checkoutservice-2.error_rate    526
18   trace.from_checkoutservice-2.duration_p95    526
19  trace.from_checkoutservice-2.duration_mean    526,    row_count              min_timestamp              max_timestamp
0       1852  2022-03-20T10:30:00+08:00  2022-03-20T11:00:00+08:00,                     cmdb_id  count
0               adservice-2     62
1             cartservice-1     62
2             cartservice-0     62
3             cartservice-2     62
4         currencyservice-1     62
5         currencyservice-0     62
6         checkoutservice-2     62
7            emailservice-1     62
8         currencyservice-2     62
9   recommendationservice-2     62
10  productcatalogservice-0     62
11         paymentservice-0     62
12               frontend-2     62
13               frontend-1     62
14               frontend-0     62
15        shippingservice-2     62
16        shippingservice-1     62
17  recommendationservice-1     62
18  recommendationservice-0     62
19              adservice-1     60,           log_name  count
0  log.error_count    926
1    log.row_count    926,    row_count min_timestamp max_timestamp
0          0          None          None, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df from kernel. Ensure 'value' numeric and drop NA values for threshold calculation
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_nonnull = metric_df.dropna(subset=['value'])

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) BEFORE any time filtering
thresholds = (
    metric_nonnull
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(P95=lambda x: x.quantile(0.95), P5=lambda x: x.quantile(0.05))
    .reset_index()
)

# Total number of unique series scanned
total_series_scanned = thresholds.shape[0]

# 2) Filter metric rows to the incident window in UTC+8: 2022-03-20 10:30:00 → 2022-03-20 11:00:00
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2022-03-20 10:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 11:00:00', tz=tz)
window_df = metric_df[(metric_df['ts'] >= start) & (metric_df['ts'] <= end)].copy()

# 3) Detect anomaly rows inside the window using the precomputed P95/P5 thresholds
# Merge thresholds onto windowed data
window_with_thresh = window_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Anomaly condition (guard against missing thresholds)
cond_high = window_with_thresh['P95'].notna() & (window_with_thresh['value'] >= window_with_thresh['P95'])
cond_low = window_with_thresh['P5'].notna() & (window_with_thresh['value'] <= window_with_thresh['P5'])
window_with_thresh['is_anomaly'] = cond_high | cond_low

# Store full per-row anomaly table for follow-up
anomaly_rows_df = window_with_thresh[window_with_thresh['is_anomaly']].copy()

# 4) Produce compact ranked table aggregated per (cmdb_id, kpi_name)
# Window-level stats (max/min in the window per series)
window_stats = (
    window_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(max_value_in_window='max', min_value_in_window='min')
    .reset_index()
)

# Anomaly-level stats (count and earliest anomaly ts)
anomaly_stats = (
    anomaly_rows_df
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(anomaly_count=('value', 'size'), earliest_anomaly_timestamp=('ts', 'min'))
    .reset_index()
)

# Combine stats with thresholds
summary = (
    anomaly_stats
    .merge(window_stats, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
)

# Compute severity_ratio = (max_value_in_window - P95)/P95 if P95>0 else None
def compute_severity(row):
    p95 = row['P95']
    maxv = row['max_value_in_window']
    if pd.isna(p95) or p95 == 0 or pd.isna(maxv):
        return None
    return (maxv - p95) / p95

summary['severity_ratio'] = summary.apply(compute_severity, axis=1)

# Format earliest timestamp to ISO string in UTC+8
summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')

# Order by anomaly_count desc and prepare compact table
ranked = (
    summary
    .sort_values(['anomaly_count', 'earliest_anomaly_timestamp'], ascending=[False, True])
    .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp',
             'max_value_in_window', 'min_value_in_window', 'P95', 'P5', 'severity_ratio']]
)

ranked_top20 = ranked.head(20)

# 5) Totals
total_anomaly_rows = anomaly_rows_df.shape[0]

# Return compact results: top 20 ranked table, total series scanned, total anomaly rows
ranked_top20, total_series_scanned, total_anomaly_rows

```
Out[2]:
```
summary = (
    f"Summary of anomaly detection (compact):\n\n"
    f"- Total (cmdb_id, kpi_name) series scanned: {total_series_scanned}\n"
    f"- Total anomaly rows detected in window 2022-03-20 10:30 → 11:00 (UTC+8): {total_anomaly_rows}\n\n"
    f"- Top-ranked series (top 20 summary): dominated by cmdb_id 'adservice' (each top entry has anomaly_count=31).\n"
    f"  Example KPI names from the top results: {', '.join(ranked_top20['kpi_name'].astype(str).tolist()[:10])}\n"
    f"  Earliest anomaly timestamp for these top entries: {ranked_top20['earliest_anomaly_timestamp'].iat[0]}\n"
    f"  Observed max/min values for these top entries equal their P95/P5 thresholds; severity_ratio is 0 (or NaN when P95 is 0),\n"
    f"  indicating the window values match the precomputed extreme percentiles rather than exceeding them.\n\n"
    f"- Notes: many of the top anomalous KPIs are Java runtime / GC and app-level counters for 'adservice'.\n"
    f"  The full per-row anomaly table is saved in the kernel variable 'anomaly_rows_df' for follow-up investigation."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window          P95           P5  severity_ratio
2   adservice                                        app.grpc.rr             31   2022-03-20T10:30:00+0800                100.0                100.0        100.0        100.0             0.0
3   adservice                                        app.grpc.sr             31   2022-03-20T10:30:00+0800                100.0                100.0        100.0        100.0             0.0
4   adservice                                     app.http.count             31   2022-03-20T10:30:00+0800                 24.0                 24.0         24.0         24.0             0.0
6   adservice                                        app.http.rr             31   2022-03-20T10:30:00+0800                100.0                100.0        100.0        100.0             0.0
7   adservice                                        app.http.sr             31   2022-03-20T10:30:00+0800                100.0                100.0        100.0        100.0             0.0
8   adservice    runtime.java_lang_ClassLoading_LoadedClassCount             31   2022-03-20T10:30:00+0800               5162.0               5162.0       5162.0       5162.0             0.0
9   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...             31   2022-03-20T10:30:00+0800               5185.0               5185.0       5185.0       5185.0             0.0
10  adservice  runtime.java_lang_ClassLoading_UnloadedClassCount             31   2022-03-20T10:30:00+0800                 23.0                 23.0         23.0         23.0             0.0
11  adservice             runtime.java_lang_ClassLoading_Verbose             31   2022-03-20T10:30:00+0800                  0.0                  0.0          0.0          0.0             NaN
12  adservice  runtime.java_lang_Compilation_CompilationTimeM...             31   2022-03-20T10:30:00+0800                  1.0                  1.0          1.0          1.0             0.0
15  adservice  runtime.java_lang_GarbageCollector_CollectionC...             31   2022-03-20T10:30:00+0800                  8.0                  8.0          8.0          8.0             0.0
17  adservice  runtime.java_lang_GarbageCollector_CollectionT...             31   2022-03-20T10:30:00+0800                480.0                480.0        480.0        480.0             0.0
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31   2022-03-20T10:30:00+0800                  1.0                  1.0          1.0          1.0             0.0
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31   2022-03-20T10:30:00+0800                  1.0                  1.0          1.0          1.0             0.0
21  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31   2022-03-20T10:30:00+0800                 94.0                 94.0         94.0         94.0             0.0
23  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31   2022-03-20T10:30:00+0800          539245384.0          539245384.0  539245384.0  539245384.0             0.0
25  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31   2022-03-20T10:30:00+0800                  8.0                  8.0          8.0          8.0             0.0
26  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31   2022-03-20T10:30:00+0800           23330816.0           23330816.0   23330816.0   23330816.0             0.0
27  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31   2022-03-20T10:30:00+0800           23330816.0           23330816.0   23330816.0   23330816.0             0.0
28  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31   2022-03-20T10:30:00+0800            3932160.0            3932160.0    3932160.0    3932160.0             0.0, 5253, 120189)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: anomaly_rows_df, thresholds
# Ensure anomaly_rows_df exists and has necessary columns
# If thresholds exists, we'll merge to be safe (thresholds has P95/P5 per series)

# Merge thresholds onto anomaly rows to ensure P95/P5 present
anomaly_with_thresh = anomaly_rows_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left', suffixes=('', '_th'))

# Group by series and compute requested aggregates (max/min computed over anomaly rows in window)
grouped = (
    anomaly_with_thresh
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        anomaly_count = ('value', 'size'),
        earliest_anomaly_timestamp = ('ts', 'min'),
        max_value_in_window = ('value', 'max'),
        min_value_in_window = ('value', 'min'),
        P95 = ('P95', 'first'),
        P5  = ('P5', 'first')
    )
)

# Compute severity ratios per instructions
def calc_sev_high(row):
    p95 = row['P95']
    maxv = row['max_value_in_window']
    if pd.isna(p95) or p95 == 0 or pd.isna(maxv):
        return None
    return (maxv - p95) / p95

def calc_sev_low(row):
    p5 = row['P5']
    minv = row['min_value_in_window']
    if pd.isna(p5) or p5 == 0 or pd.isna(minv):
        return None
    # denominator uses abs(P5)
    return (p5 - minv) / abs(p5)

grouped['severity_ratio_high'] = grouped.apply(calc_sev_high, axis=1)
grouped['severity_ratio_low']  = grouped.apply(calc_sev_low, axis=1)

# Format earliest timestamp to ISO string in UTC+8
grouped['earliest_anomaly_timestamp'] = pd.to_datetime(grouped['earliest_anomaly_timestamp']).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')

# Store grouped table in kernel variable for follow-up
grouped_anomaly_summary = grouped.copy()

# 2a) Top 20 by severity_ratio_high DESC (exclude nulls)
top_high = (
    grouped_anomaly_summary[ grouped_anomaly_summary['severity_ratio_high'].notna() ]
    .sort_values('severity_ratio_high', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'P95', 'severity_ratio_high']]
    .head(20)
)

# 2b) Top 20 by severity_ratio_low DESC (exclude nulls)
top_low = (
    grouped_anomaly_summary[ grouped_anomaly_summary['severity_ratio_low'].notna() ]
    .sort_values('severity_ratio_low', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'min_value_in_window', 'P5', 'severity_ratio_low']]
    .head(20)
)

# 3) Summary counts for severity > 0.5
number_of_series_with_severity_high_gt_0_5 = (grouped_anomaly_summary['severity_ratio_high'] > 0.5).sum()
number_of_series_with_severity_low_gt_0_5  = (grouped_anomaly_summary['severity_ratio_low'] > 0.5).sum()

# Return compact outputs
top_high, top_low, number_of_series_with_severity_high_gt_0_5, number_of_series_with_severity_low_gt_0_5

```
Out[3]:
```
Summary of grouped anomaly severities (compact):

- Counts:
  - Number of series with severity_ratio_high > 0.5: 111
  - Number of series with severity_ratio_low > 0.5: 17

- Top examples by high-side severity (severity_ratio_high, max vs P95):
  1. paymentservice2-0 — container.node-6.container_network_receive_MB... — severity ~20.42 (max 0.6013 vs P95 0.02807), earliest anomaly 2022-03-20T10:34+08:00
  2. emailservice-2 — container.node-6.container_network_receive_MB... — severity ~19.23 (max 0.6089 vs P95 0.03010), earliest 10:33
  3. shippingservice2-0 — container.node-6.container_network_receive_MB... — severity ~13.97, earliest 10:31
  4. node-1 — system.io.r_s — severity ~13.10 (max 204.5 vs P95 14.5), earliest 10:30
  5. redis-cart2-0 — container.node-6.container_network_receive_MB... — severity ~11.92, earliest 10:38
  - (Top 20 list is dominated by container network receive MB metrics across many services and some node I/O metrics. Anomaly counts per series are small—often between 2 and 29 rows—occurring around 10:30–10:44 UTC+8.)

- Top examples by low-side severity (severity_ratio_low, P5 vs min):
  1. shippingservice-2 — container.node-5.container_cpu_system_seconds — severity_ratio_low = 1.0 (min 0 vs P5 0.003333), earliest 10:31
  2. node-1 — system.io.w_await — severity_ratio_low = 1.0 (min 0 vs P5 0.23), earliest 10:30
  3. shippingservice-2 — container.node-5.container_cpu_user_seconds — severity_ratio_low = 1.0 (min 0 vs P5 0.005), earliest 10:30
  4. shippingservice-1 — container.node-6.container_cpu_system_seconds — severity_ratio_low = 1.0 (min 0 vs P5 0.005), earliest 10:35
  5. node-1 / node-3 — several system.io.* metrics (w_s, await, svctm, util) with severity_ratio_low = 1.0 (min 0), earliest 10:30–10:33
  - (Many low-side top entries are metrics dropping to zero while their historical P5 > 0, producing high relative severity.)

- Interpretation / notes:
  - High-side anomalies are largely large jumps in container network receive MB for many services (e.g., paymentservice2-0, emailservice-2, shippingservice2-0), plus node I/O spikes (node-1 read/sec). These show max values many times their historical P95.
  - Low-side anomalies commonly reflect metrics falling to zero (or much lower than usual), e.g., CPU seconds or I/O waits dropping to 0 while P5 > 0, yielding severity_ratio_low = 1.0.
  - Earliest anomalies mostly start at the incident window start (≈ 10:30 UTC+8) with many at 10:30–10:38.

- Data availability:
  - The full grouped per-series summary is stored in the kernel variable grouped_anomaly_summary.
  - The full per-row anomaly table is stored as anomaly_rows_df for follow-up inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window         P95  severity_ratio_high
3996   paymentservice2-0  container.node-6.container_network_receive_MB....              2   2022-03-20T10:34:00+0800             0.601284    0.028068            20.422718
2526      emailservice-2  container.node-6.container_network_receive_MB....              4   2022-03-20T10:33:00+0800             0.608891    0.030095            19.232246
5198  shippingservice2-0  container.node-6.container_network_receive_MB....              3   2022-03-20T10:31:00+0800             0.615532    0.041119            13.969490
3410              node-1                                      system.io.r_s             21   2022-03-20T10:30:00+0800           204.500000   14.500000            13.103448
4902       redis-cart2-0  container.node-6.container_network_receive_MB....              2   2022-03-20T10:38:00+0800             0.624016    0.048311            11.916578
3590              node-4                                      system.io.w_s             29   2022-03-20T10:30:00+0800            12.500000    1.000000            11.500000
3411              node-1                                    system.io.rkb_s             21   2022-03-20T10:30:00+0800          5026.000000  416.000000            11.081731
1902  checkoutservice2-0  container.node-6.container_network_receive_MB....              5   2022-03-20T10:32:00+0800             0.627870    0.058346             9.761100
2452      emailservice-1  container.node-6.container_network_receive_MB....              5   2022-03-20T10:38:00+0800             0.329032    0.033106             8.938786
475          adservice-1  container.node-6.container_network_receive_MB....              4   2022-03-20T10:38:00+0800             0.644251    0.066453             8.694768
2378      emailservice-0  container.node-6.container_network_receive_MB....              5   2022-03-20T10:34:00+0800             0.317932    0.035562             7.940318
381          adservice-0  container.node-6.container_network_receive_MB....              3   2022-03-20T10:42:00+0800             0.654968    0.074453             7.797022
5118   shippingservice-2  container.node-5.container_network_receive_MB....              4   2022-03-20T10:32:00+0800             0.418978    0.047926             7.742145
3912    paymentservice-2  container.node-6.container_memory_failures.con...              8   2022-03-20T10:30:00+0800           709.500000   82.500000             7.600000
3914    paymentservice-2  container.node-6.container_memory_failures.hie...              8   2022-03-20T10:30:00+0800           709.500000   82.500000             7.600000
1490   checkoutservice-0  container.node-6.container_network_receive_MB....              2   2022-03-20T10:37:00+0800             0.628913    0.073836             7.517724
5038   shippingservice-1  container.node-6.container_network_receive_MB....              4   2022-03-20T10:33:00+0800             0.349416    0.044581             6.837785
1386      cartservice2-0  container.node-5.container_network_receive_MB....              5   2022-03-20T10:32:00+0800             0.649734    0.084064             6.728996
1190       cartservice-1  container.node-6.container_network_receive_MB....              3   2022-03-20T10:40:00+0800             0.645360    0.088570             6.286436
1090       cartservice-0  container.node-6.container_network_receive_MB....              4   2022-03-20T10:38:00+0800             0.649496    0.089739             6.237625,                       cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  min_value_in_window         P5  severity_ratio_low
5082        shippingservice-2      container.node-5.container_cpu_system_seconds              5   2022-03-20T10:31:00+0800             0.000000   0.003333            1.000000
3414                   node-1                                  system.io.w_await              4   2022-03-20T10:30:00+0800             0.000000   0.230000            1.000000
5084        shippingservice-2        container.node-5.container_cpu_user_seconds              6   2022-03-20T10:30:00+0800             0.000000   0.005000            1.000000
5002        shippingservice-1      container.node-6.container_cpu_system_seconds             14   2022-03-20T10:35:00+0800             0.000000   0.005000            1.000000
3415                   node-1                                      system.io.w_s              4   2022-03-20T10:30:00+0800             0.000000   3.000000            1.000000
3531                   node-3                                      system.io.w_s              5   2022-03-20T10:32:00+0800             0.000000   5.000000            1.000000
3530                   node-3                                  system.io.w_await              6   2022-03-20T10:32:00+0800             0.000000   0.290000            1.000000
3529                   node-3                                     system.io.util              5   2022-03-20T10:32:00+0800             0.000000   0.350000            1.000000
3524                   node-3                                    system.io.await              6   2022-03-20T10:32:00+0800             0.000000   0.290000            1.000000
3528                   node-3                                    system.io.svctm              5   2022-03-20T10:33:00+0800             0.000000   0.170000            1.000000
1452        checkoutservice-0  container.node-6.container_cpu_cfs_throttled_s...              5   2022-03-20T10:30:00+0800             0.002902   0.045102            0.935663
4264  productcatalogservice-1  mesh.source.productcatalogservice.jaeger-colle...              5   2022-03-20T10:33:00+0800             0.000525   0.002100            0.750000
1588        checkoutservice-1  container.node-6.container_cpu_cfs_throttled_s...              3   2022-03-20T10:33:00+0800             0.003820   0.011983            0.681176
1868       checkoutservice2-0        container.node-6.container_cpu_user_seconds              6   2022-03-20T10:30:00+0800             0.005000   0.015000            0.666667
3421                   node-1                         system.mem.real.pct_useage              2   2022-03-20T10:41:00+0800            21.830000  53.170000            0.589430
3645                   node-5                                      system.load.1              5   2022-03-20T10:36:00+0800             0.110000   0.260000            0.576923
1866       checkoutservice2-0      container.node-6.container_cpu_system_seconds              3   2022-03-20T10:44:00+0800             0.005000   0.010000            0.500000
1182            cartservice-1  container.node-6.container_memory_failures.hie...             11   2022-03-20T10:33:00+0800            16.500000  33.000000            0.500000
1180            cartservice-1  container.node-6.container_memory_failures.con...             11   2022-03-20T10:33:00+0800            16.500000  33.000000            0.500000
4924        shippingservice-0        container.node-6.container_cpu_user_seconds              6   2022-03-20T10:31:00+0800             0.005000   0.010000            0.500000, np.int64(111), np.int64(17))```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse kernel variables: anomaly_rows_df, grouped_anomaly_summary
tz = pytz.timezone('Asia/Shanghai')

# 1) Select candidate series where severity_ratio_high > 0.5 OR severity_ratio_low > 0.5
candidates = grouped_anomaly_summary[
    (grouped_anomaly_summary['severity_ratio_high'] > 0.5) |
    (grouped_anomaly_summary['severity_ratio_low'] > 0.5)
].loc[:, ['cmdb_id', 'kpi_name', 'P95', 'P5', 'severity_ratio_high', 'severity_ratio_low']].copy()

number_of_candidate_series = candidates.shape[0]

# If no candidates, return empty structures
if number_of_candidate_series == 0:
    faults_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','seq_id','fault_start_timestamp','fault_end_timestamp',
        'fault_length_in_points','max_value_in_sequence','min_value_in_sequence',
        'P95','P5','severity_ratio_high','severity_ratio_low'
    ])
    top_high_faults = faults_df.head(0)
    top_low_faults = faults_df.head(0)
    total_number_of_faults_detected_across_candidates = 0
else:
    # 2) For each selected series, order anomaly rows by timestamp and identify consecutive sequences
    # Filter anomaly rows to only candidate series
    candidate_rows = anomaly_rows_df.merge(
        candidates[['cmdb_id','kpi_name']],
        on=['cmdb_id','kpi_name'],
        how='inner'
    ).copy()
    
    # Ensure ts is datetime with tz
    candidate_rows['ts'] = pd.to_datetime(candidate_rows['ts']).dt.tz_convert(tz)
    
    # Define function to assign sequence ids within each series where gap > 60s starts new seq
    def assign_seq(df):
        df = df.sort_values('ts').copy()
        # gap in seconds
        df['gap'] = df['ts'].diff().dt.total_seconds().fillna(0)
        df['seq_id'] = (df['gap'] > 60).cumsum()
        return df

    candidate_seq = candidate_rows.groupby(['cmdb_id','kpi_name'], group_keys=False).apply(assign_seq).reset_index(drop=True)
    
    # Aggregate each sequence to produce a fault record
    faults = (
        candidate_seq
        .groupby(['cmdb_id','kpi_name','seq_id'], as_index=False)
        .agg(
            fault_start_timestamp = ('ts', 'min'),
            fault_end_timestamp = ('ts', 'max'),
            fault_length_in_points = ('value', 'size'),
            max_value_in_sequence = ('value', 'max'),
            min_value_in_sequence = ('value', 'min')
        )
    )

    # Merge series-level thresholds and severity ratios from candidates
    faults = faults.merge(candidates, on=['cmdb_id','kpi_name'], how='left')

    # Compute/ensure severity ratios present (they came from candidates)
    # Format timestamps to UTC+8 ISO strings
    faults['fault_start_timestamp'] = pd.to_datetime(faults['fault_start_timestamp']).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    faults['fault_end_timestamp']   = pd.to_datetime(faults['fault_end_timestamp']).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')

    # Store full faults table in kernel variable
    faults_df = faults.copy()

    # 3a) Top 20 faults ordered by severity_ratio_high DESC (only where >0)
    top_high_faults = (
        faults_df[ faults_df['severity_ratio_high'].notna() & (faults_df['severity_ratio_high'] > 0) ]
        .sort_values('severity_ratio_high', ascending=False)
        .loc[:, ['cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
                 'fault_length_in_points','max_value_in_sequence','P95','severity_ratio_high']]
        .head(20)
    )

    # 3b) Top 20 faults ordered by severity_ratio_low DESC (only where >0)
    top_low_faults = (
        faults_df[ faults_df['severity_ratio_low'].notna() & (faults_df['severity_ratio_low'] > 0) ]
        .sort_values('severity_ratio_low', ascending=False)
        .loc[:, ['cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
                 'fault_length_in_points','min_value_in_sequence','P5','severity_ratio_low']]
        .head(20)
    )

    total_number_of_faults_detected_across_candidates = faults_df.shape[0]

# Return compact outputs (DataFrames and totals)
top_high_faults, top_low_faults, number_of_candidate_series, total_number_of_faults_detected_across_candidates

```
Out[4]:
```
Summary of fault sequences (compact):

- Totals:
  - Candidate series (severity_high>0.5 OR severity_low>0.5): 126
  - Total fault sequences detected across candidates: 444
  - Full faults table saved as kernel variable: faults_df

- Top high-side faults (examples from top 20):
  - paymentservice2-0 — container.node-6.container_network_receive_MB...  
    - two fault sequences at 2022-03-20T10:34:00+08:00 and 10:58:00+08:00 (each 1 point)  
    - max in sequence up to 0.6013 vs P95 0.02807 → severity_ratio_high ≈ 20.42
  - emailservice-2 — container.node-6.container_network_receive_MB...  
    - several single-point sequences (e.g., 10:33, 10:42, 10:44, 10:47) with max up to 0.6089 vs P95 0.03010 → severity ≈ 19.23
  - shippingservice2-0 — container.node-6.container_network_receive_MB... (single-point sequences, severity ≈ 13.97)
  - node-1 — system.io.r_s  
    - multiple sequences including a 7-point sequence 2022-03-20T10:34 → 10:40 with max 204.5 vs P95 14.5 → severity ≈ 13.10  
    - other short sequences at 10:30–11:00 where values dropped to 0 (also flagged)
  - redis-cart2-0 — container.node-6.container_network_receive_MB... (single-point sequences, severity ≈ 11.92)

  - Note: top-high list is dominated by container network receive MB spikes across many services and notable node I/O spikes. Many high-severity faults are short (single-point) but with values far above historical P95.

- Top low-side faults (examples from top 20):
  - node-3 — multiple system.io.* metrics (system.io.util, w_await, w_s, await, svctm)  
    - several sequences where metric dropped to 0 while P5 > 0 → severity_ratio_low = 1.0 (min 0 vs P5>0)
  - shippingservice-2 — container.node-5.container_cpu_system_seconds / container_cpu_user_seconds  
    - sequences with min 0 vs small positive P5 → severity_ratio_low = 1.0
  - node-1 — system.io.w_await  
    - sequences with min 0 vs P5 0.23 → severity_ratio_low = 1.0
  - Other low-side top faults include short sequences where CPU or I/O metrics fell to zero or much lower than historical P5.

  - Note: low-side top list is mainly metrics dropping to zero (or much below P5), yielding high relative severity; these are often single- or few-point sequences.

- Interpretation / next steps:
  - High-severity candidates indicate large transient spikes in container network receive traffic for many services (paymentservice2-0, emailservice-2, shippingservice2-0, redis-cart2-0, etc.) and a notable multi-point node read I/O spike on node-1 (system.io.r_s).
  - Low-severity-high (severity_ratio_low) candidates indicate sudden drops to zero in CPU or I/O metrics on nodes/containers (node-1, node-3, shippingservice-2), which may reflect metric collection interruptions, ephemeral process pauses, or actual resource starvation/stop.
  - Use faults_df to correlate these fault sequences with trace/log events (timestamps around 10:30–10:40 and scattered up to 11:00 UTC+8) to determine root cause (network spikes vs node I/O issues vs metric collection gaps).

The original code execution output of IPython Kernel is also provided below for reference:

(                cmdb_id                                           kpi_name     fault_start_timestamp       fault_end_timestamp  fault_length_in_points  max_value_in_sequence        P95  severity_ratio_high
337   paymentservice2-0  container.node-6.container_network_receive_MB....  2022-03-20T10:58:00+0800  2022-03-20T10:58:00+0800                       1               0.028125   0.028068            20.422718
336   paymentservice2-0  container.node-6.container_network_receive_MB....  2022-03-20T10:34:00+0800  2022-03-20T10:34:00+0800                       1               0.601284   0.028068            20.422718
122      emailservice-2  container.node-6.container_network_receive_MB....  2022-03-20T10:47:00+0800  2022-03-20T10:47:00+0800                       1               0.022603   0.030095            19.232246
121      emailservice-2  container.node-6.container_network_receive_MB....  2022-03-20T10:44:00+0800  2022-03-20T10:44:00+0800                       1               0.022130   0.030095            19.232246
119      emailservice-2  container.node-6.container_network_receive_MB....  2022-03-20T10:33:00+0800  2022-03-20T10:33:00+0800                       1               0.608891   0.030095            19.232246
120      emailservice-2  container.node-6.container_network_receive_MB....  2022-03-20T10:42:00+0800  2022-03-20T10:42:00+0800                       1               0.030149   0.030095            19.232246
442  shippingservice2-0  container.node-6.container_network_receive_MB....  2022-03-20T10:37:00+0800  2022-03-20T10:37:00+0800                       1               0.615532   0.041119            13.969490
441  shippingservice2-0  container.node-6.container_network_receive_MB....  2022-03-20T10:31:00+0800  2022-03-20T10:31:00+0800                       1               0.027020   0.041119            13.969490
443  shippingservice2-0  container.node-6.container_network_receive_MB....  2022-03-20T10:47:00+0800  2022-03-20T10:47:00+0800                       1               0.041542   0.041119            13.969490
180              node-1                                      system.io.r_s  2022-03-20T10:50:00+0800  2022-03-20T10:50:00+0800                       1               0.000000  14.500000            13.103448
178              node-1                                      system.io.r_s  2022-03-20T10:44:00+0800  2022-03-20T10:46:00+0800                       3               0.000000  14.500000            13.103448
181              node-1                                      system.io.r_s  2022-03-20T10:52:00+0800  2022-03-20T10:52:00+0800                       1               0.000000  14.500000            13.103448
183              node-1                                      system.io.r_s  2022-03-20T10:57:00+0800  2022-03-20T10:57:00+0800                       1               0.000000  14.500000            13.103448
179              node-1                                      system.io.r_s  2022-03-20T10:48:00+0800  2022-03-20T10:48:00+0800                       1               0.000000  14.500000            13.103448
177              node-1                                      system.io.r_s  2022-03-20T10:34:00+0800  2022-03-20T10:40:00+0800                       7             204.500000  14.500000            13.103448
182              node-1                                      system.io.r_s  2022-03-20T10:54:00+0800  2022-03-20T10:55:00+0800                       2               0.000000  14.500000            13.103448
176              node-1                                      system.io.r_s  2022-03-20T10:30:00+0800  2022-03-20T10:32:00+0800                       3               0.000000  14.500000            13.103448
184              node-1                                      system.io.r_s  2022-03-20T10:59:00+0800  2022-03-20T11:00:00+0800                       2               0.000000  14.500000            13.103448
400       redis-cart2-0  container.node-6.container_network_receive_MB....  2022-03-20T11:00:00+0800  2022-03-20T11:00:00+0800                       1               0.049200   0.048311            11.916578
399       redis-cart2-0  container.node-6.container_network_receive_MB....  2022-03-20T10:38:00+0800  2022-03-20T10:38:00+0800                       1               0.624016   0.048311            11.916578,                cmdb_id                                       kpi_name     fault_start_timestamp       fault_end_timestamp  fault_length_in_points  min_value_in_sequence        P5  severity_ratio_low
240             node-3                                 system.io.util  2022-03-20T10:32:00+0800  2022-03-20T10:33:00+0800                       2               0.000000  0.350000                 1.0
241             node-3                                 system.io.util  2022-03-20T10:54:00+0800  2022-03-20T10:55:00+0800                       2               3.550000  0.350000                 1.0
242             node-3                                 system.io.util  2022-03-20T10:59:00+0800  2022-03-20T10:59:00+0800                       1               3.700000  0.350000                 1.0
243             node-3                              system.io.w_await  2022-03-20T10:32:00+0800  2022-03-20T10:33:00+0800                       2               0.000000  0.290000                 1.0
244             node-3                              system.io.w_await  2022-03-20T10:35:00+0800  2022-03-20T10:35:00+0800                       1               0.920000  0.290000                 1.0
245             node-3                              system.io.w_await  2022-03-20T10:41:00+0800  2022-03-20T10:41:00+0800                       1               0.290000  0.290000                 1.0
246             node-3                              system.io.w_await  2022-03-20T10:46:00+0800  2022-03-20T10:46:00+0800                       1               0.760000  0.290000                 1.0
247             node-3                              system.io.w_await  2022-03-20T10:56:00+0800  2022-03-20T10:56:00+0800                       1               0.760000  0.290000                 1.0
248             node-3                                  system.io.w_s  2022-03-20T10:32:00+0800  2022-03-20T10:33:00+0800                       2               0.000000  5.000000                 1.0
249             node-3                                  system.io.w_s  2022-03-20T10:35:00+0800  2022-03-20T10:35:00+0800                       1              57.500000  5.000000                 1.0
250             node-3                                  system.io.w_s  2022-03-20T10:46:00+0800  2022-03-20T10:46:00+0800                       1              72.500000  5.000000                 1.0
251             node-3                                  system.io.w_s  2022-03-20T10:55:00+0800  2022-03-20T10:55:00+0800                       1              46.000000  5.000000                 1.0
428  shippingservice-2  container.node-5.container_cpu_system_seconds  2022-03-20T10:31:00+0800  2022-03-20T10:31:00+0800                       1               0.000000  0.003333                 1.0
431  shippingservice-2  container.node-5.container_cpu_system_seconds  2022-03-20T10:53:00+0800  2022-03-20T10:54:00+0800                       2               0.015000  0.003333                 1.0
430  shippingservice-2  container.node-5.container_cpu_system_seconds  2022-03-20T10:49:00+0800  2022-03-20T10:49:00+0800                       1               0.003333  0.003333                 1.0
196             node-1                              system.io.w_await  2022-03-20T10:30:00+0800  2022-03-20T10:30:00+0800                       1               2.120000  0.230000                 1.0
197             node-1                              system.io.w_await  2022-03-20T10:36:00+0800  2022-03-20T10:36:00+0800                       1               0.000000  0.230000                 1.0
198             node-1                              system.io.w_await  2022-03-20T10:45:00+0800  2022-03-20T10:45:00+0800                       1               1.870000  0.230000                 1.0
199             node-1                              system.io.w_await  2022-03-20T11:00:00+0800  2022-03-20T11:00:00+0800                       1               0.230000  0.230000                 1.0
234             node-3                                system.io.await  2022-03-20T10:46:00+0800  2022-03-20T10:46:00+0800                       1               0.760000  0.290000                 1.0, 126, 444)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# reuse trace_df and tz if present; else load/define
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
tz = pytz.timezone('Asia/Shanghai')

# Ensure 'value' numeric
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) BEFORE filtering by time
trace_nonnull = trace_df.dropna(subset=['value'])
trace_thresholds = (
    trace_nonnull
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(P95=lambda x: x.quantile(0.95), P5=lambda x: x.quantile(0.05))
    .reset_index()
)

total_trace_series_scanned = trace_thresholds.shape[0]

# 2) Filter trace rows to incident window (UTC+8)
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2022-03-20 10:30:00', tz=tz)
end   = pd.Timestamp('2022-03-20 11:00:00', tz=tz)
trace_window = trace_df[(trace_df['ts'] >= start) & (trace_df['ts'] <= end)].copy()

# 3) Detect anomaly rows within window using precomputed thresholds
trace_with_thresh = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
cond_high = trace_with_thresh['P95'].notna() & (trace_with_thresh['value'] >= trace_with_thresh['P95'])
cond_low  = trace_with_thresh['P5'].notna()  & (trace_with_thresh['value'] <= trace_with_thresh['P5'])
trace_with_thresh['is_anomaly'] = cond_high | cond_low

# Store full per-row anomalies for follow-up
trace_anomalies_df = trace_with_thresh[trace_with_thresh['is_anomaly']].copy()

total_trace_anomaly_rows_detected = trace_anomalies_df.shape[0]

# 4) Group anomalies by (cmdb_id, trace_name) and compute requested aggregates
if trace_anomalies_df.shape[0] > 0:
    grouped_trace_anom = (
        trace_anomalies_df
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(
            anomaly_count = ('value', 'size'),
            earliest_anomaly_timestamp = ('ts', 'min'),
            max_value_in_window = ('value', 'max'),
            min_value_in_window = ('value', 'min'),
            P95 = ('P95', 'first'),
            P5  = ('P5', 'first')
        )
    )

    def calc_sev_high(r):
        p95 = r['P95']; maxv = r['max_value_in_window']
        if pd.isna(p95) or p95 == 0 or pd.isna(maxv): return None
        return (maxv - p95) / p95

    def calc_sev_low(r):
        p5 = r['P5']; minv = r['min_value_in_window']
        if pd.isna(p5) or p5 == 0 or pd.isna(minv): return None
        return (p5 - minv) / abs(p5)

    grouped_trace_anom['severity_ratio_high'] = grouped_trace_anom.apply(calc_sev_high, axis=1)
    grouped_trace_anom['severity_ratio_low']  = grouped_trace_anom.apply(calc_sev_low, axis=1)

    # Format earliest timestamp to ISO with tz
    grouped_trace_anom['earliest_anomaly_timestamp'] = pd.to_datetime(grouped_trace_anom['earliest_anomaly_timestamp']).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')

    # 5a) Top 20 by severity_ratio_high DESC (only where >0)
    top_trace_high = (
        grouped_trace_anom[ grouped_trace_anom['severity_ratio_high'].notna() & (grouped_trace_anom['severity_ratio_high'] > 0) ]
        .sort_values('severity_ratio_high', ascending=False)
        .loc[:, ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','P95','severity_ratio_high']]
        .head(20)
    )

    # 5b) Top 20 by severity_ratio_low DESC (only where >0)
    top_trace_low = (
        grouped_trace_anom[ grouped_trace_anom['severity_ratio_low'].notna() & (grouped_trace_anom['severity_ratio_low'] > 0) ]
        .sort_values('severity_ratio_low', ascending=False)
        .loc[:, ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','min_value_in_window','P5','severity_ratio_low']]
        .head(20)
    )
else:
    # No anomalies found in window
    grouped_trace_anom = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','min_value_in_window','P95','P5','severity_ratio_high','severity_ratio_low'])
    top_trace_high = grouped_trace_anom.head(0)
    top_trace_low  = grouped_trace_anom.head(0)

# Return compact outputs: top high, top low, totals; also store trace_anomalies_df in kernel
top_trace_high, top_trace_low, total_trace_series_scanned, total_trace_anomaly_rows_detected

```
Out[5]:
```
Summary of trace anomaly detection (compact):

- Totals:
  - Trace series scanned (unique (cmdb_id, trace_name)): 1,232
  - Trace anomaly rows detected in window 2022-03-20 10:30 → 11:00 (UTC+8): 17,066

- Top high-side trace anomalies (examples — top 20 dominated by increased latencies):
  - frontend-1 — trace.to_cartservice-2.duration_mean: anomaly_count=4, earliest=2022-03-20T10:32+08:00, max=0.002962, P95=0.000218, severity ≈ 12.56
  - cartservice-2 — trace.from_frontend-1.duration_mean: anomaly_count=4, earliest=10:32, max=0.002962, P95=0.000218, severity ≈ 12.56
  - cartservice-2 / frontend-2 pairs (from/to cartservice duration_mean) with severity ~10.0
  - frontend-0 / frontend-2 / cartservice-0 / cartservice-1 many duration_mean traces with severity 5–8× above P95
  - checkoutservice-1 trace.from_root.duration_p95 and several payment/currency/checkout traces with 3–7× severity

  Interpretation: many short-duration but large-relative increases in trace durations (frontend → cartservice and cartservice → frontend flows), indicating elevated latency in the request path among frontend and cartservice components during the incident window.

- Top low-side trace anomalies (examples — top 20 dominated by drops to zero or much lower than P5):
  - Many trace.to_cartservice/trace.from_frontend series (frontend and cartservice) report min_value = 0 while P5 > 0 → severity_ratio_low = 1.0 (i.e., dropped to zero).
  - checkoutservice2-0 / cartservice2-0 traces show many zero-row or zero-duration events (severity 1.0).
  - frontend2-0 trace.self.row_count dropped sharply (min 1 vs P5 583.5 → severity ≈ 0.998), indicating sudden loss of trace row counts.
  - Several trace row_count metrics for checkout/currency/cartservice dropped from expected counts to 1 (or lower), severity 0.5–1.0.

  Interpretation: numerous traces have dropped to zero or much lower than historical minima (possible tainted telemetry, dropped spans, sampling change, or real drops in traffic).

- Overall takeaway and suggested next steps:
  - Trace anomalies point to two correlated symptoms during 10:30–11:00 UTC+8: (A) elevated request latencies between frontend and cartservice (high-side severity), and (B) many traces/row counts dropping to near-zero (low-side severity).
  - Correlate these trace faults with the previously produced faults_df (network receive MB spikes and node I/O anomalies) — especially the frontend↔cartservice flow and affected nodes (node-1/node-3) — to identify whether latency spikes align with network/node I/O faults or with missing trace data.
  - Focus initial investigation on frontend and cartservice interactions around ~10:30–10:40 and on nodes showing I/O/network anomalies.

- Data availability:
  - Full per-row trace anomaly table saved as: trace_anomalies_df
  - Use trace_anomalies_df and faults_df for timestamp-aligned correlation and root-cause analysis.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                  trace_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window       P95  severity_ratio_high
688         frontend-1        trace.to_cartservice-2.duration_mean              4   2022-03-20T10:32:00+0800             0.002962  0.000218            12.562271
112      cartservice-2         trace.from_frontend-1.duration_mean              4   2022-03-20T10:32:00+0800             0.002962  0.000218            12.562271
116      cartservice-2         trace.from_frontend-2.duration_mean              4   2022-03-20T10:37:00+0800             0.002357  0.000213            10.042399
780         frontend-2        trace.to_cartservice-2.duration_mean              4   2022-03-20T10:37:00+0800             0.002357  0.000213            10.042399
592         frontend-0        trace.to_cartservice-1.duration_mean              5   2022-03-20T10:34:00+0800             0.002103  0.000221             8.526614
80       cartservice-1         trace.from_frontend-0.duration_mean              5   2022-03-20T10:34:00+0800             0.002103  0.000221             8.526614
772         frontend-2        trace.to_cartservice-0.duration_mean              5   2022-03-20T10:33:00+0800             0.003071  0.000340             8.035010
60       cartservice-0         trace.from_frontend-2.duration_mean              5   2022-03-20T10:33:00+0800             0.003071  0.000340             8.035010
108      cartservice-2         trace.from_frontend-0.duration_mean              4   2022-03-20T10:30:00+0800             0.002182  0.000244             7.942623
596         frontend-0        trace.to_cartservice-2.duration_mean              4   2022-03-20T10:30:00+0800             0.002182  0.000244             7.942623
684         frontend-1        trace.to_cartservice-1.duration_mean              4   2022-03-20T10:35:00+0800             0.001652  0.000200             7.249688
84       cartservice-1         trace.from_frontend-1.duration_mean              4   2022-03-20T10:35:00+0800             0.001652  0.000200             7.249688
241  checkoutservice-1                trace.from_root.duration_p95              4   2022-03-20T10:31:00+0800             0.031797  0.003968             7.013660
64       cartservice-0                    trace.self.duration_mean             27   2022-03-20T10:30:00+0800             0.000933  0.000133             6.022958
588         frontend-0        trace.to_cartservice-0.duration_mean              4   2022-03-20T10:30:00+0800             0.001217  0.000191             5.386775
52       cartservice-0         trace.from_frontend-0.duration_mean              4   2022-03-20T10:30:00+0800             0.001217  0.000191             5.386775
888   paymentservice-0  trace.from_checkoutservice-2.duration_mean              5   2022-03-20T10:41:00+0800             0.000723  0.000160             3.518750
376  checkoutservice-2     trace.to_paymentservice-0.duration_mean              5   2022-03-20T10:41:00+0800             0.000723  0.000160             3.518750
501  currencyservice-2   trace.from_checkoutservice-1.duration_p95              4   2022-03-20T10:33:00+0800             0.000731  0.000162             3.499846
269  checkoutservice-1     trace.to_currencyservice-2.duration_p95              4   2022-03-20T10:33:00+0800             0.000731  0.000162             3.499846,                 cmdb_id                                   trace_name  anomaly_count earliest_anomaly_timestamp  min_value_in_window          P5  severity_ratio_low
688          frontend-1         trace.to_cartservice-2.duration_mean              4   2022-03-20T10:32:00+0800             0.000000    0.000034            1.000000
684          frontend-1         trace.to_cartservice-1.duration_mean              4   2022-03-20T10:35:00+0800             0.000000    0.000037            1.000000
596          frontend-0         trace.to_cartservice-2.duration_mean              4   2022-03-20T10:30:00+0800             0.000000    0.000017            1.000000
588          frontend-0         trace.to_cartservice-0.duration_mean              4   2022-03-20T10:30:00+0800             0.000000    0.000035            1.000000
112       cartservice-2          trace.from_frontend-1.duration_mean              4   2022-03-20T10:32:00+0800             0.000000    0.000034            1.000000
52        cartservice-0          trace.from_frontend-0.duration_mean              4   2022-03-20T10:30:00+0800             0.000000    0.000035            1.000000
124      cartservice2-0  trace.from_checkoutservice2-0.duration_mean             19   2022-03-20T10:30:00+0800             0.000000    0.000167            1.000000
424  checkoutservice2-0        trace.to_cartservice2-0.duration_mean             19   2022-03-20T10:30:00+0800             0.000000    0.000167            1.000000
425  checkoutservice2-0         trace.to_cartservice2-0.duration_p95             24   2022-03-20T10:30:00+0800             0.000000    0.000750            1.000000
780          frontend-2         trace.to_cartservice-2.duration_mean              4   2022-03-20T10:37:00+0800             0.000000    0.000016            1.000000
84        cartservice-1          trace.from_frontend-1.duration_mean              4   2022-03-20T10:35:00+0800             0.000000    0.000037            1.000000
108       cartservice-2          trace.from_frontend-0.duration_mean              4   2022-03-20T10:30:00+0800             0.000000    0.000017            1.000000
116       cartservice-2          trace.from_frontend-2.duration_mean              4   2022-03-20T10:37:00+0800             0.000000    0.000016            1.000000
125      cartservice2-0   trace.from_checkoutservice2-0.duration_p95             24   2022-03-20T10:30:00+0800             0.000000    0.000750            1.000000
851         frontend2-0                         trace.self.row_count              4   2022-03-20T10:44:00+0800             1.000000  583.500000            0.998286
849         frontend2-0                      trace.self.duration_p95              4   2022-03-20T10:40:00+0800             0.001835    0.007293            0.748389
848         frontend2-0                     trace.self.duration_mean              4   2022-03-20T10:41:00+0800             0.001835    0.004422            0.585029
499   currencyservice-2       trace.from_checkoutservice-0.row_count             11   2022-03-20T10:30:00+0800             1.000000    2.000000            0.500000
363   checkoutservice-2         trace.to_currencyservice-2.row_count             10   2022-03-20T10:32:00+0800             1.000000    2.000000            0.500000
507   currencyservice-2       trace.from_checkoutservice-2.row_count             10   2022-03-20T10:32:00+0800             1.000000    2.000000            0.500000, 1232, 17066)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# reuse kernel variables: faults_df, trace_anomalies_df
tz = pytz.timezone('Asia/Shanghai')

# Ensure necessary variables exist
try:
    faults_df
    trace_anomalies_df
except NameError as e:
    raise RuntimeError("Required kernel variables 'faults_df' or 'trace_anomalies_df' not found.") from e

# Ensure trace_anomalies_df has tz-aware 'ts' and faults_df has start/end as tz-aware datetimes
trace_anomalies_df['ts'] = pd.to_datetime(trace_anomalies_df['ts']).dt.tz_convert(tz)

# Parse fault start/end timestamps into tz-aware datetimes (they may currently be strings)
faults_df['_fault_start_dt'] = pd.to_datetime(faults_df['fault_start_timestamp']).dt.tz_convert(tz)
faults_df['_fault_end_dt']   = pd.to_datetime(faults_df['fault_end_timestamp']).dt.tz_convert(tz)

# Pre-calc arrays for trace times to speed repeated filtering
trace_times = trace_anomalies_df['ts'].values
trace_names = trace_anomalies_df['trace_name'].values

# Prepare list to collect correlation records
records = []
for idx, row in faults_df.reset_index().iterrows():
    fa_cmdb = row['cmdb_id']
    fa_kpi  = row['kpi_name']
    fa_start = row['_fault_start_dt']
    fa_end   = row['_fault_end_dt']
    window_start = fa_start - pd.Timedelta(seconds=60)
    window_end   = fa_end   + pd.Timedelta(seconds=60)
    # boolean mask of trace anomalies within window
    mask = (trace_anomalies_df['ts'] >= window_start) & (trace_anomalies_df['ts'] <= window_end)
    overlap_df = trace_anomalies_df[mask]
    overlap_count = int(overlap_df.shape[0])
    if overlap_count > 0:
        earliest_overlap_ts = pd.to_datetime(overlap_df['ts'].min()).tz_convert(tz).strftime('%Y-%m-%dT%H:%M:%S%z')
        example_names = ";".join(overlap_df['trace_name'].drop_duplicates().head(5).astype(str).tolist())
    else:
        earliest_overlap_ts = None
        example_names = ""
    records.append({
        'cmdb_id': fa_cmdb,
        'kpi_name': fa_kpi,
        'fault_start_timestamp': fa_start.strftime('%Y-%m-%dT%H:%M:%S%z'),
        'fault_end_timestamp': fa_end.strftime('%Y-%m-%dT%H:%M:%S%z'),
        'fault_length_in_points': int(row['fault_length_in_points']),
        'severity_ratio_high': row.get('severity_ratio_high', None),
        'severity_ratio_low': row.get('severity_ratio_low', None),
        'overlap_count': overlap_count,
        'earliest_overlapping_trace_timestamp': earliest_overlap_ts,
        'example_overlapping_trace_names': example_names
    })

# Build correlation DataFrame and store in kernel variable
fault_trace_correlation_df = pd.DataFrame.from_records(records)

# Totals
total_faults_scanned = faults_df.shape[0]
total_trace_anomaly_rows = trace_anomalies_df.shape[0]

# 2) Produce compact ranked table (top 20) ordered by overlap_count DESC
ranked_top20 = (
    fault_trace_correlation_df
    .sort_values(['overlap_count', 'severity_ratio_high'], ascending=[False, False])
    .loc[:, ['cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
             'fault_length_in_points','severity_ratio_high','severity_ratio_low',
             'overlap_count','earliest_overlapping_trace_timestamp','example_overlapping_trace_names']]
    .head(20)
)

# Return compact outputs
ranked_top20, total_faults_scanned, total_trace_anomaly_rows

```
Out[6]:
```
Summary of fault ↔ trace correlation (compact):

- Totals:
  - Fault sequences scanned: 444
  - Trace anomaly rows (window): 17,066
  - Full correlation table saved as: fault_trace_correlation_df

- Top correlated faults (highest overlap_count — examples):
  1. shippingservice-1 — container memory failure metrics (two KPIs)  
     - Fault window: 2022-03-20T10:30 → 2022-03-20T11:00 (31 points)  
     - severity_ratio_high ≈ 4.0, overlap_count = 17,066 (earliest overlapping trace 2022-03-20T10:30)  
     - Example overlapping traces: trace.from_frontend-0.duration_p95; trace.from_frontend-0.duration_mean; ...
  2. emailservice2-0 — mesh/jaeger collector KPI  
     - Fault window: 10:30 → 11:00 (31 pts), severity_ratio_high ≈ 2.90, overlap_count = 17,066
  3. paymentservice-1 / paymentservice-2 — container CPU system seconds  
     - Fault window: 10:30 → 11:00 (31 pts), severity_ratio_high = 1.0, overlap_count = 17,066
  4. Several other long-duration faults (checkoutservice-2 memory failures, cpu cfs throttling across currency/payment services, node-5 CPU iowait, node-4 I/O/w_s, etc.) also show large overlap counts (10k–17k), with earliest overlapping traces at ~10:30.
  5. Shorter faults with substantial overlaps (e.g., node-1 avg_q_sz, node-4 net retrans_segs, shippingservice-2 mesh entries) show overlap counts in the 9k–11k range with earliest overlaps around 10:32–10:36.

- Interpretation:
  - The highest overlap_counts are for faults that span almost the entire incident window (10:30–11:00). Because many trace anomalies occur throughout that window, these long faults naturally overlap many trace-anomaly rows — producing very high counts.
  - Example overlapping trace names are dominated by frontend→service latency metrics (e.g., trace.from_frontend-0.duration_p95/mean), which supports a correlation between service/node faults and elevated trace latencies / reduced trace row counts.
  - Shorter but high-severity faults (e.g., network receive MB spikes or node I/O spikes) also overlap trace anomalies roughly in the same timeframe (10:30–10:40), and may be causal.

- Recommended next steps:
  - Prioritize investigation of faults that both have high severity and high overlap (shippingservice-1 memory failures, emailservice2-0, paymentservice nodes, node-4/node-5 I/O/network metrics).  
  - Use the saved tables (faults_df, trace_anomalies_df, fault_trace_correlation_df) to drill down: inspect trace names/timestamps listed in example_overlapping_trace_names and sample trace rows around the earliest overlapping trace timestamps to determine causality (e.g., network/I/O spike → increased frontend→cartservice latency).

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                           kpi_name     fault_start_timestamp       fault_end_timestamp  fault_length_in_points  severity_ratio_high  severity_ratio_low  overlap_count earliest_overlapping_trace_timestamp                    example_overlapping_trace_names
413        shippingservice-1  container.node-6.container_memory_failures.con...  2022-03-20T10:30:00+0800  2022-03-20T11:00:00+0800                      31             4.000000                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
414        shippingservice-1  container.node-6.container_memory_failures.hie...  2022-03-20T10:30:00+0800  2022-03-20T11:00:00+0800                      31             4.000000                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
129          emailservice2-0  mesh.source.emailservice2.jaeger-collector.ist...  2022-03-20T10:30:00+0800  2022-03-20T11:00:00+0800                      31             2.904762                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
309         paymentservice-1      container.node-6.container_cpu_system_seconds  2022-03-20T10:30:00+0800  2022-03-20T11:00:00+0800                      31             1.000000                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
310         paymentservice-2      container.node-6.container_cpu_system_seconds  2022-03-20T10:30:00+0800  2022-03-20T11:00:00+0800                      31             1.000000                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
76         checkoutservice-2  container.node-5.container_memory_failures.con...  2022-03-20T10:30:00+0800  2022-03-20T11:00:00+0800                      31             1.000000                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
77         checkoutservice-2  container.node-5.container_memory_failures.hie...  2022-03-20T10:30:00+0800  2022-03-20T11:00:00+0800                      31             1.000000                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
94         currencyservice-1  container.node-6.container_cpu_cfs_throttled_p...  2022-03-20T10:30:00+0800  2022-03-20T11:00:00+0800                      31             1.000000                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
100        currencyservice-2  container.node-6.container_cpu_cfs_throttled_p...  2022-03-20T10:30:00+0800  2022-03-20T11:00:00+0800                      31             1.000000                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
274                   node-5                                  system.cpu.iowait  2022-03-20T10:30:00+0800  2022-03-20T11:00:00+0800                      31             1.000000                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
95         currencyservice-1  container.node-6.container_cpu_cfs_throttled_s...  2022-03-20T10:30:00+0800  2022-03-20T11:00:00+0800                      31             0.958082                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
108       currencyservice2-0  container.node-6.container_cpu_cfs_throttled_s...  2022-03-20T10:30:00+0800  2022-03-20T10:59:00+0800                      30             0.806520                 NaN          17066             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
168                   node-1                                 system.io.avg_q_sz  2022-03-20T10:37:00+0800  2022-03-20T11:00:00+0800                      24             2.040000                 NaN          13550             2022-03-20T10:36:00+0800  trace.from_frontend-0.duration_mean;trace.from...
133               frontend-1  container.node-5.container_memory_failures.con...  2022-03-20T10:30:00+0800  2022-03-20T10:50:00+0800                      21             1.000000                 NaN          12494             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
135               frontend-1  container.node-5.container_memory_failures.hie...  2022-03-20T10:30:00+0800  2022-03-20T10:50:00+0800                      21             1.000000                 NaN          12494             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
268                   node-4                                      system.io.w_s  2022-03-20T10:41:00+0800  2022-03-20T11:00:00+0800                      20            11.500000                 NaN          11345             2022-03-20T10:40:00+0800  trace.from_frontend-0.error_rate;trace.from_fr...
265                   node-4                                     system.io.util  2022-03-20T10:41:00+0800  2022-03-20T11:00:00+0800                      20             1.000000                 NaN          11345             2022-03-20T10:40:00+0800  trace.from_frontend-0.error_rate;trace.from_fr...
439        shippingservice-2  mesh.destination.checkoutservice.shippingservi...  2022-03-20T10:30:00+0800  2022-03-20T10:48:00+0800                      19             1.000000                 NaN          11320             2022-03-20T10:30:00+0800  trace.from_frontend-0.duration_p95;trace.from_...
270                   node-4                        system.net.tcp.retrans_segs  2022-03-20T10:33:00+0800  2022-03-20T10:49:00+0800                      17             3.600000                 NaN          10713             2022-03-20T10:32:00+0800  trace.from_frontend-0.error_rate;trace.from_fr...
362  productcatalogservice-2  mesh.destination.recommendationservice2.produc...  2022-03-20T10:37:00+0800  2022-03-20T10:51:00+0800                      15             1.000000                 NaN           9520             2022-03-20T10:36:00+0800  trace.from_frontend-0.duration_mean;trace.from..., 444, 17066)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: faults_df, trace_anomalies_df, grouped_trace_anom (if present)
# Ensure they exist
try:
    faults_df
    trace_anomalies_df
except NameError as e:
    raise RuntimeError("Required kernel variables 'faults_df' or 'trace_anomalies_df' not found in kernel.") from e

# Ensure trace_anomalies_df has tz-aware 'ts'
trace_anomalies_df['ts'] = pd.to_datetime(trace_anomalies_df['ts']).dt.tz_convert(tz)

# Ensure faults_df has tz-aware parsed start/end datetimes (create if missing)
if '_fault_start_dt' not in faults_df.columns or '_fault_end_dt' not in faults_df.columns:
    faults_df['_fault_start_dt'] = pd.to_datetime(faults_df['fault_start_timestamp']).dt.tz_convert(tz)
    faults_df['_fault_end_dt']   = pd.to_datetime(faults_df['fault_end_timestamp']).dt.tz_convert(tz)

# 1) Extract node-1 system.io.r_s fault rows (all columns)
node1_io_faults = faults_df[
    (faults_df['cmdb_id'] == 'node-1') &
    (faults_df['kpi_name'].str.contains('system.io.r_s', na=False))
].copy()

# 2) Choose preferred interval: prefer the 7-point sequence starting at 2022-03-20T10:34:00+08:00 if present,
# otherwise pick the node-1 io fault with largest severity_ratio_high
preferred_fault = None
if not node1_io_faults.empty:
    # try to find the specific 7-point sequence with the requested start
    target_start = pd.Timestamp('2022-03-20 10:34:00', tz=tz)
    mask7 = (node1_io_faults['fault_length_in_points'] == 7) & (node1_io_faults['_fault_start_dt'] == target_start)
    if mask7.any():
        preferred_fault = node1_io_faults[mask7].iloc[0]
    else:
        # fallback: choose the sequence with max severity_ratio_high (prefer non-null)
        node1_io_faults['sev_high_filled'] = node1_io_faults['severity_ratio_high'].fillna(-1)
        preferred_fault = node1_io_faults.sort_values('sev_high_filled', ascending=False).iloc[0]
        node1_io_faults = node1_io_faults.drop(columns=['sev_high_filled'], errors='ignore')
else:
    preferred_fault = None

# If preferred_fault found, build slack window and filter trace anomalies
if preferred_fault is not None:
    fa_start = preferred_fault['_fault_start_dt']
    fa_end   = preferred_fault['_fault_end_dt']
    window_start = fa_start - pd.Timedelta(seconds=60)
    window_end   = fa_end   + pd.Timedelta(seconds=60)

    # Filter trace anomalies within the slack window
    overlapping_traces = trace_anomalies_df[(trace_anomalies_df['ts'] >= window_start) & (trace_anomalies_df['ts'] <= window_end)].copy()

    # Determine anomaly side per-row using P95/P5 (per-row has P95/P5 from earlier merge)
    def side_label(r):
        high = pd.notna(r['P95']) and r['value'] >= r['P95']
        low  = pd.notna(r['P5'])  and r['value'] <= r['P5']
        if high and low:
            return 'both'
        if high:
            return 'high'
        if low:
            return 'low'
        return 'unknown'

    overlapping_traces['anomaly_side'] = overlapping_traces.apply(side_label, axis=1)

    # Try to attach per-series severity ratios from grouped_trace_anom if present in kernel
    if 'grouped_trace_anom' in globals():
        sev_map = grouped_trace_anom.set_index(['cmdb_id','trace_name'])[['severity_ratio_high','severity_ratio_low']]
        overlapping_traces = overlapping_traces.merge(sev_map, left_on=['cmdb_id','trace_name'], right_index=True, how='left')
    else:
        # Fallback: compute per-series severity ratios from trace_anomalies_df grouping (max/min vs P95/P5)
        temp = overlapping_traces.groupby(['cmdb_id','trace_name']).agg(
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min'),
            P95=('P95','first'),
            P5=('P5','first')
        ).reset_index()
        def calc_high(r):
            if pd.isna(r['P95']) or r['P95']==0: return None
            return (r['max_value_in_window'] - r['P95'])/r['P95']
        def calc_low(r):
            if pd.isna(r['P5']) or r['P5']==0: return None
            return (r['P5'] - r['min_value_in_window'])/abs(r['P5'])
        temp['severity_ratio_high'] = temp.apply(calc_high, axis=1)
        temp['severity_ratio_low']  = temp.apply(calc_low, axis=1)
        overlapping_traces = overlapping_traces.merge(temp[['cmdb_id','trace_name','severity_ratio_high','severity_ratio_low']],
                                                    on=['cmdb_id','trace_name'], how='left')

    # Prepare compact sample ordered by timestamp (earliest_anomaly_timestamp column as requested)
    overlapping_traces['earliest_anomaly_timestamp'] = overlapping_traces['ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    selected_trace_sample = overlapping_traces.sort_values('ts').loc[:, [
        'earliest_anomaly_timestamp','cmdb_id','trace_name','value','anomaly_side','severity_ratio_high','severity_ratio_low'
    ]].head(20).copy()

    # Aggregate counts of overlapping trace anomalies grouped by cmdb_id (top 10)
    trace_counts_by_cmdb = overlapping_traces.groupby('cmdb_id').size().reset_index(name='overlap_count').sort_values('overlap_count', ascending=False).head(10)
else:
    # No preferred fault found -> empty outputs
    selected_trace_sample = pd.DataFrame(columns=['earliest_anomaly_timestamp','cmdb_id','trace_name','value','anomaly_side','severity_ratio_high','severity_ratio_low'])
    trace_counts_by_cmdb = pd.DataFrame(columns=['cmdb_id','overlap_count'])
    window_start = window_end = fa_start = fa_end = None

# 3) Find container-level metric faults mapped to node-1 whose fault_start within [fault_start -60s, fault_end +60s]
node1_container_faults = pd.DataFrame(columns=['cmdb_id','kpi_name','fault_start_timestamp','fault_length_in_points','severity_ratio_high','severity_ratio_low'])
if preferred_fault is not None:
    # match kpi_name containing 'container.node-1' OR kpi_name containing 'container.node-1' pattern
    mask_container_node1 = faults_df['kpi_name'].str.contains('container.node-1', na=False)
    # Also include series where kpi_name contains 'container.node-6' etc mapped to node-1? The instruction prefers kpi_name contains 'container.node-1'
    # Filter by fault start within slack window
    mask_time = (faults_df['_fault_start_dt'] >= window_start) & (faults_df['_fault_start_dt'] <= window_end)
    node1_container_faults = faults_df[mask_container_node1 & mask_time].loc[:, ['cmdb_id','kpi_name','fault_start_timestamp','fault_length_in_points','severity_ratio_high','severity_ratio_low']].copy().head(20)

# Store results in kernel variables for follow-up
node1_io_faults_df = node1_io_faults.copy()
preferred_node1_fault = preferred_fault.copy() if preferred_fault is not None else None
overlapping_traces_df = overlapping_traces.copy() if preferred_fault is not None else pd.DataFrame()
selected_trace_sample_df = selected_trace_sample.copy()
trace_counts_by_cmdb_df = trace_counts_by_cmdb.copy()
node1_container_faults_df = node1_container_faults.copy()

# Return compact outputs
node1_io_faults_df, preferred_node1_fault, selected_trace_sample_df, trace_counts_by_cmdb_df, node1_container_faults_df

```
Out[7]:
```
Findings for node-1 I/O fault correlation (compact):

1) node-1 system.io.r_s fault rows (all sequences)
- Multiple fault sequences detected for cmdb_id = node-1, kpi_name = system.io.r_s:
  - 2022-03-20T10:30 → 10:32  (3 points)
  - 2022-03-20T10:34 → 10:40  (7 points)  ← preferred (largest severity)
  - 2022-03-20T10:44 → 10:46  (3 points)
  - 2022-03-20T10:48 → 10:48  (1 point)
  - 2022-03-20T10:50 → 10:50  (1 point)
  - 2022-03-20T10:52 → 10:52  (1 point)
  - 2022-03-20T10:54 → 10:55  (2 points)
  - 2022-03-20T10:57 → 10:57  (1 point)
  - 2022-03-20T10:59 → 11:00  (2 points)

2) Confirmed preferred interval (largest severity)
- Chosen interval: 2022-03-20T10:34:00+08:00 → 2022-03-20T10:40:00+08:00 (7 points)
- Stats for that sequence: max value = 204.5, min = 0.0, P95 = 14.5, severity_ratio_high ≈ 13.10.

3) Trace anomalies overlapping the preferred interval (±60s slack)
- Filter window used: 2022-03-20T10:33:00 → 2022-03-20T10:41:00 (UTC+8).
- Compact sample (top 20 ordered by timestamp). Examples include (timestamp, cmdb_id, trace_name, value, side, per-trace severity):
  - 2022-03-20T10:33:00 — adservice-0 — trace.from_frontend-0.error_rate — 0.0 — both
  - 2022-03-20T10:33:00 — frontend-2 — trace.to_cartservice-2.duration_p95 — 0.001 — high
  - 2022-03-20T10:33:00 — frontend-2 — trace.to_cartservice-1.duration_p95 — 0.001 — high
  - 2022-03-20T10:33:00 — frontend-2 — many trace.to_* error_rate rows — both
  - 2022-03-20T10:33:00 — frontend-2 — trace.to_cartservice-0.duration_mean — 0.0 — low (severity ~8.035)
  - ... (sample shows mix of high-side duration spikes and low-side drops to 0 in trace row_count / error_rate)
- Per-cmdb_id aggregate counts of overlapping trace anomalies (top 10):
  - checkoutservice-1: 421
  - checkoutservice-2: 416
  - checkoutservice-0: 396
  - frontend-0: 358
  - frontend-1: 354
  - frontend-2: 349
  - cartservice-0: 171
  - cartservice-1: 167
  - cartservice-2: 165
  - productcatalogservice-1: 159

Interpretation of trace overlap:
- The preferred node-1 read I/O spike (max 204.5, severity ~13× P95) coincides with many trace anomalies in frontend → checkout/cartservice paths around ~10:33–10:37.
- Trace anomalies are mixed: some show high-side latency increases (duration_p95/duration_mean) and many show low-side drops (row_count or error_rate=0), suggesting both increased latency and missing/reduced tracing events during the same window.

4) Container-level faults mapped to node-1 in that window
- No container-level faults with kpi_name containing 'container.node-1' were found with start timestamps inside the node-1 preferred slack window (result is empty).

Stored kernel variables for follow-up:
- node1_io_faults_df (all node-1 system.io.r_s sequences)
- preferred_node1_fault (the 7-point 10:34→10:40 sequence)
- overlapping_traces_df (all trace anomaly rows in the slack window)
- selected_trace_sample_df (sample returned above)
- trace_counts_by_cmdb_df (per-cmdb_id counts)
- node1_container_faults_df (empty for this query)

Recommended next steps:
- Investigate node-1 I/O subsystem (disk device, queue depth, avg_q_sz) around 10:34–10:40 since a multi-point read I/O spike aligns with elevated frontend→service latency and many trace anomalies.
- Correlate with node-level metrics such as system.io.avg_q_sz, io.wait, and network retransmits (some node-4 and node-5 faults were present earlier) and with logs on node-1 for I/O errors during 10:34–10:40.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id       kpi_name  seq_id     fault_start_timestamp       fault_end_timestamp  fault_length_in_points  max_value_in_sequence  min_value_in_sequence   P95   P5  severity_ratio_high  severity_ratio_low           _fault_start_dt             _fault_end_dt
176  node-1  system.io.r_s       0  2022-03-20T10:30:00+0800  2022-03-20T10:32:00+0800                       3                    0.0                    0.0  14.5  0.0            13.103448                 NaN 2022-03-20 10:30:00+08:00 2022-03-20 10:32:00+08:00
177  node-1  system.io.r_s       1  2022-03-20T10:34:00+0800  2022-03-20T10:40:00+0800                       7                  204.5                    0.0  14.5  0.0            13.103448                 NaN 2022-03-20 10:34:00+08:00 2022-03-20 10:40:00+08:00
178  node-1  system.io.r_s       2  2022-03-20T10:44:00+0800  2022-03-20T10:46:00+0800                       3                    0.0                    0.0  14.5  0.0            13.103448                 NaN 2022-03-20 10:44:00+08:00 2022-03-20 10:46:00+08:00
179  node-1  system.io.r_s       3  2022-03-20T10:48:00+0800  2022-03-20T10:48:00+0800                       1                    0.0                    0.0  14.5  0.0            13.103448                 NaN 2022-03-20 10:48:00+08:00 2022-03-20 10:48:00+08:00
180  node-1  system.io.r_s       4  2022-03-20T10:50:00+0800  2022-03-20T10:50:00+0800                       1                    0.0                    0.0  14.5  0.0            13.103448                 NaN 2022-03-20 10:50:00+08:00 2022-03-20 10:50:00+08:00
181  node-1  system.io.r_s       5  2022-03-20T10:52:00+0800  2022-03-20T10:52:00+0800                       1                    0.0                    0.0  14.5  0.0            13.103448                 NaN 2022-03-20 10:52:00+08:00 2022-03-20 10:52:00+08:00
182  node-1  system.io.r_s       6  2022-03-20T10:54:00+0800  2022-03-20T10:55:00+0800                       2                    0.0                    0.0  14.5  0.0            13.103448                 NaN 2022-03-20 10:54:00+08:00 2022-03-20 10:55:00+08:00
183  node-1  system.io.r_s       7  2022-03-20T10:57:00+0800  2022-03-20T10:57:00+0800                       1                    0.0                    0.0  14.5  0.0            13.103448                 NaN 2022-03-20 10:57:00+08:00 2022-03-20 10:57:00+08:00
184  node-1  system.io.r_s       8  2022-03-20T10:59:00+0800  2022-03-20T11:00:00+0800                       2                    0.0                    0.0  14.5  0.0            13.103448                 NaN 2022-03-20 10:59:00+08:00 2022-03-20 11:00:00+08:00, cmdb_id                                      node-1
kpi_name                              system.io.r_s
seq_id                                            1
fault_start_timestamp      2022-03-20T10:34:00+0800
fault_end_timestamp        2022-03-20T10:40:00+0800
fault_length_in_points                            7
max_value_in_sequence                         204.5
min_value_in_sequence                           0.0
P95                                            14.5
P5                                              0.0
severity_ratio_high                       13.103448
severity_ratio_low                              NaN
_fault_start_dt           2022-03-20 10:34:00+08:00
_fault_end_dt             2022-03-20 10:40:00+08:00
Name: 177, dtype: object,      earliest_anomaly_timestamp      cmdb_id                                   trace_name     value anomaly_side  severity_ratio_high  severity_ratio_low
3610   2022-03-20T10:33:00+0800  adservice-0             trace.from_frontend-0.error_rate  0.000000         both                  NaN                 NaN
4390   2022-03-20T10:33:00+0800   frontend-2  trace.to_productcatalogservice-0.error_rate  0.000000         both                  NaN                 NaN
4386   2022-03-20T10:33:00+0800   frontend-2        trace.to_currencyservice-2.error_rate  0.000000         both                  NaN                 NaN
4382   2022-03-20T10:33:00+0800   frontend-2        trace.to_currencyservice-1.error_rate  0.000000         both                  NaN                 NaN
4378   2022-03-20T10:33:00+0800   frontend-2        trace.to_currencyservice-0.error_rate  0.000000         both                  NaN                 NaN
4375   2022-03-20T10:33:00+0800   frontend-2         trace.to_checkoutservice-2.row_count  1.000000          low             0.000000                 0.0
4374   2022-03-20T10:33:00+0800   frontend-2        trace.to_checkoutservice-2.error_rate  0.000000         both                  NaN                 NaN
4371   2022-03-20T10:33:00+0800   frontend-2         trace.to_checkoutservice-1.row_count  1.000000          low             0.000000                 0.0
4370   2022-03-20T10:33:00+0800   frontend-2        trace.to_checkoutservice-1.error_rate  0.000000         both                  NaN                 NaN
4367   2022-03-20T10:33:00+0800   frontend-2         trace.to_checkoutservice-0.row_count  1.000000          low             0.000000                 0.0
4366   2022-03-20T10:33:00+0800   frontend-2        trace.to_checkoutservice-0.error_rate  0.000000         both                  NaN                 NaN
4362   2022-03-20T10:33:00+0800   frontend-2            trace.to_cartservice-2.error_rate  0.000000         both                  NaN                 NaN
4361   2022-03-20T10:33:00+0800   frontend-2          trace.to_cartservice-2.duration_p95  0.001000         high             0.000000                 NaN
4358   2022-03-20T10:33:00+0800   frontend-2            trace.to_cartservice-1.error_rate  0.000000         both                  NaN                 NaN
4357   2022-03-20T10:33:00+0800   frontend-2          trace.to_cartservice-1.duration_p95  0.001000         high             0.000000                 NaN
4394   2022-03-20T10:33:00+0800   frontend-2  trace.to_productcatalogservice-1.error_rate  0.000000         both                  NaN                 NaN
4354   2022-03-20T10:33:00+0800   frontend-2            trace.to_cartservice-0.error_rate  0.000000         both                  NaN                 NaN
4352   2022-03-20T10:33:00+0800   frontend-2         trace.to_cartservice-0.duration_mean  0.000000          low             8.035010                 NaN
4350   2022-03-20T10:33:00+0800   frontend-2              trace.to_adservice-2.error_rate  0.000000         both                  NaN                 NaN
4348   2022-03-20T10:33:00+0800   frontend-2           trace.to_adservice-2.duration_mean  0.000014          low             0.160221                 0.0,                     cmdb_id  overlap_count
9         checkoutservice-1            421
10        checkoutservice-2            416
8         checkoutservice-0            396
20               frontend-0            358
21               frontend-1            354
22               frontend-2            349
4             cartservice-0            171
5             cartservice-1            167
6             cartservice-2            165
29  productcatalogservice-1            159, Empty DataFrame
Columns: [cmdb_id, kpi_name, fault_start_timestamp, fault_length_in_points, severity_ratio_high, severity_ratio_low]
Index: [])```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# reuse faults_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# define window bounds (UTC+8)
window_start = pd.Timestamp('2022-03-20T10:30:00', tz=tz)
window_end   = pd.Timestamp('2022-03-20T11:00:00', tz=tz)

# ensure faults_df has tz-aware _fault_start_dt/_fault_end_dt
if '_fault_start_dt' not in faults_df.columns or '_fault_end_dt' not in faults_df.columns:
    faults_df['_fault_start_dt'] = pd.to_datetime(faults_df['fault_start_timestamp']).dt.tz_convert(tz)
    faults_df['_fault_end_dt']   = pd.to_datetime(faults_df['fault_end_timestamp']).dt.tz_convert(tz)
else:
    # coerce to tz-aware if needed
    faults_df['_fault_start_dt'] = pd.to_datetime(faults_df['_fault_start_dt']).dt.tz_convert(tz)
    faults_df['_fault_end_dt']   = pd.to_datetime(faults_df['_fault_end_dt']).dt.tz_convert(tz)

# filter for node-1 faults with fault_start in the given window
node1_faults_window_df = faults_df[
    (faults_df['cmdb_id'] == 'node-1') &
    (faults_df['_fault_start_dt'] >= window_start) &
    (faults_df['_fault_start_dt'] <= window_end)
].copy()

# select requested columns and limit to 20 rows for compact display
node1_faults_window_df = node1_faults_window_df.loc[:, [
    'kpi_name','fault_start_timestamp','fault_end_timestamp','fault_length_in_points',
    'max_value_in_sequence','min_value_in_sequence','P95','P5','severity_ratio_high','severity_ratio_low'
]]

# aggregates: distinct kpi_name count and earliest fault_start_timestamp
distinct_kpi_count = int(node1_faults_window_df['kpi_name'].nunique())
if node1_faults_window_df.shape[0] > 0:
    earliest_start_dt = pd.to_datetime(faults_df.loc[
        (faults_df['cmdb_id']=='node-1') &
        (faults_df['_fault_start_dt'] >= window_start) &
        (faults_df['_fault_start_dt'] <= window_end),
        '_fault_start_dt'
    ]).min()
    earliest_fault_start_timestamp = earliest_start_dt.tz_convert(tz).strftime('%Y-%m-%dT%H:%M:%S%z')
else:
    earliest_fault_start_timestamp = None

# Store result in kernel variable and return compact outputs
node1_faults_window_df, distinct_kpi_count, earliest_fault_start_timestamp

```
Out[8]:
```
Summary of node-1 faults in 2022-03-20T10:30 → 11:00 (UTC+8):

- Fault sequences found for node-1 in this window: 63 sequences (node1_faults_window_df saved in kernel).
- Distinct KPI series for node-1 in the window: 16.
- Earliest fault_start_timestamp among them: 2022-03-20T10:30:00+08:00.

Example KPI types observed (samples):
- system.io.r_s (read/sec) — multiple sequences including the 7-point 10:34 → 10:40 spike (max 204.5, P95 14.5, severity_high ≈ 13.10).
- system.cpu.iowait
- system.cpu.pct_usage
- system.net.tcp.retrans_segs
- system.io.avg_q_sz
(Full list of KPIs and all sequences are in node1_faults_window_df.)

Returned table columns (per sequence): kpi_name, fault_start_timestamp, fault_end_timestamp, fault_length_in_points, max_value_in_sequence, min_value_in_sequence, P95, P5, severity_ratio_high, severity_ratio_low.

The original code execution output of IPython Kernel is also provided below for reference:

(                        kpi_name     fault_start_timestamp       fault_end_timestamp  fault_length_in_points  max_value_in_sequence  min_value_in_sequence    P95    P5  severity_ratio_high  severity_ratio_low
158            system.cpu.iowait  2022-03-20T10:31:00+0800  2022-03-20T10:31:00+0800                       1                   0.08                   0.08   8.63  0.08             2.721900                 0.0
159            system.cpu.iowait  2022-03-20T10:33:00+0800  2022-03-20T10:33:00+0800                       1                   0.08                   0.08   8.63  0.08             2.721900                 0.0
160            system.cpu.iowait  2022-03-20T10:59:00+0800  2022-03-20T11:00:00+0800                       2                  32.12                   0.08   8.63  0.08             2.721900                 0.0
161         system.cpu.pct_usage  2022-03-20T10:34:00+0800  2022-03-20T10:34:00+0800                       1                   5.18                   5.18  32.58  5.18             0.523634                 0.0
162         system.cpu.pct_usage  2022-03-20T10:47:00+0800  2022-03-20T10:47:00+0800                       1                  32.58                  32.58  32.58  5.18             0.523634                 0.0
..                           ...                       ...                       ...                     ...                    ...                    ...    ...   ...                  ...                 ...
216  system.net.tcp.retrans_segs  2022-03-20T10:40:00+0800  2022-03-20T10:47:00+0800                       8                   0.13                   0.00   0.13  0.00             2.153846                 NaN
217  system.net.tcp.retrans_segs  2022-03-20T10:49:00+0800  2022-03-20T10:50:00+0800                       2                   0.00                   0.00   0.13  0.00             2.153846                 NaN
218  system.net.tcp.retrans_segs  2022-03-20T10:52:00+0800  2022-03-20T10:52:00+0800                       1                   0.00                   0.00   0.13  0.00             2.153846                 NaN
219  system.net.tcp.retrans_segs  2022-03-20T10:56:00+0800  2022-03-20T10:56:00+0800                       1                   0.00                   0.00   0.13  0.00             2.153846                 NaN
220  system.net.tcp.retrans_segs  2022-03-20T10:59:00+0800  2022-03-20T10:59:00+0800                       1                   0.41                   0.41   0.13  0.00             2.153846                 NaN

[63 rows x 10 columns], 16, '2022-03-20T10:30:00+0800')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# reuse log_df if present in kernel
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric values
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 1) Global P95 and P5 per (cmdb_id, log_name) BEFORE time filtering
log_nonnull = log_df.dropna(subset=['value'])
log_thresholds = (
    log_nonnull
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(P95=lambda x: x.quantile(0.95), P5=lambda x: x.quantile(0.05))
    .reset_index()
)

total_series_scanned = log_thresholds.shape[0]

# 2) Filter log rows to the incident window (UTC+8)
log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2022-03-20 10:30:00', tz=tz)
end   = pd.Timestamp('2022-03-20 11:00:00', tz=tz)
log_window = log_df[(log_df['ts'] >= start) & (log_df['ts'] <= end)].copy()

# 3) Detect anomaly rows in window using precomputed thresholds
log_with_thresh = log_window.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')
cond_high = log_with_thresh['P95'].notna() & (log_with_thresh['value'] >= log_with_thresh['P95'])
cond_low  = log_with_thresh['P5'].notna()  & (log_with_thresh['value'] <= log_with_thresh['P5'])
log_with_thresh['is_anomaly'] = cond_high | cond_low

# Store full per-row log anomalies for follow-up
log_anomalies_df = log_with_thresh[log_with_thresh['is_anomaly']].copy()

total_anomaly_rows = log_anomalies_df.shape[0]

# 4) Aggregate per-series: anomaly_count and earliest anomaly ts from anomalies; max/min in window from full window
anomaly_stats = (
    log_anomalies_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(anomaly_count=('value','size'), earliest_anomaly_timestamp=('ts','min'))
)

window_stats = (
    log_window
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(max_value_in_window=('value','max'), min_value_in_window=('value','min'))
)

summary = (
    anomaly_stats
    .merge(window_stats, on=['cmdb_id','log_name'], how='left')
    .merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
)

# severity_ratio = (max_value_in_window - P95)/P95 if P95>0 else None
def compute_severity(row):
    p95 = row['P95']
    maxv = row['max_value_in_window']
    if pd.isna(p95) or p95 == 0 or pd.isna(maxv):
        return None
    return (maxv - p95) / p95

summary['severity_ratio'] = summary.apply(compute_severity, axis=1)
summary['earliest_anomaly_timestamp'] = pd.to_datetime(summary['earliest_anomaly_timestamp']).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')

# 4) Compact ranked table ordered by anomaly_count desc (top 20)
ranked_top20 = (
    summary
    .sort_values(['anomaly_count','earliest_anomaly_timestamp'], ascending=[False, True])
    .loc[:, ['cmdb_id','log_name','anomaly_count','earliest_anomaly_timestamp',
             'max_value_in_window','min_value_in_window','P95','P5','severity_ratio']]
    .head(20)
)

# Return compact outputs (store full per-row anomalies in log_anomalies_df)
ranked_top20, total_series_scanned, total_anomaly_rows

```
Out[9]:
```
Summary of log anomaly scan (compact):

- Method: computed global P95/P5 per (cmdb_id, log_name) across full log.csv, then flagged window anomalies in 2022-03-20 10:30 → 11:00 (UTC+8).

- Totals:
  - (cmdb_id, log_name) series scanned: 62
  - Log anomaly rows detected in window: 1,069
  - Full per-row anomaly table saved as: log_anomalies_df

- Top findings (top 20 by anomaly_count):
  - The top anomalies are almost entirely log.error_count zeros across many services (examples: adservice-2, cartservice-0/1/2, checkoutservice-2, currencyservice-*, frontend-0/1/2, paymentservice-0, productcatalogservice-0, recommendationservice-*, shippingservice-*).
  - Many series show anomaly_count = 31 (i.e., flagged for most/all minutes in the window), some 30 for slightly shorter coverage. Earliest anomaly timestamps are 2022-03-20T10:30:00+08:00.
  - max/min in-window values are 0.0 and historical P95/P5 are 0.0 for these series, so the computed severity_ratio is NaN (cannot meaningfully measure relative spike when P95 == 0).

- Interpretation:
  - The log anomalies are dominated by log.error_count being zero during the incident window for many services. Because the historical P95/P5 are also zero, these are not "spikes above baseline" but rather values equal to historical extremes; severity ratio is not informative.
  - This pattern may indicate uniformly zeroized error counts (no errors logged) across services during the window or a change in log aggregation/sampling. It does not by itself indicate increased errors; check raw logs and aggregation pipeline for drops or instrumentation changes.

- Next steps:
  - Inspect log_anomalies_df and raw error logs around 10:30–11:00 for any collection/sampling issues or confirm whether error_count=0 is expected.
  - Correlate with faults_df and trace_anomalies_df (already saved) to see if zeroed log.error_count aligns with the trace and metric faults (e.g., missing telemetry or traffic drops).

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id         log_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  P95   P5  severity_ratio
4               adservice-2  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
6             cartservice-0  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
8             cartservice-1  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
10            cartservice-2  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
16        checkoutservice-2  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
18        currencyservice-0  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
20        currencyservice-1  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
22        currencyservice-2  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
26           emailservice-1  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
30               frontend-0  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
32               frontend-1  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
34               frontend-2  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
36         paymentservice-0  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
42  productcatalogservice-0  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
48  recommendationservice-0  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
50  recommendationservice-1  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
52  recommendationservice-2  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
58        shippingservice-1  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
60        shippingservice-2  log.error_count             31   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN
0               adservice-0  log.error_count             30   2022-03-20T10:30:00+0800                  0.0                  0.0  0.0  0.0             NaN, 62, 1069)```
```